# War and conflict

In [ ]:
import polars as pl

# cfg = pl.Config()
# cfg.set_tbl_rows(30)
# cfg.set_fmt_str_lengths(1000)

## Get data

In [ ]:
df = pl.read_parquet('../data/facts/FIN/*.parquet')

In [ ]:
df.schema

In [ ]:
# Number of rows
df.height

In [ ]:
# Annotate session year
df = df.with_columns(pl.col('session_date').dt.year().alias('session_year'))

# Number of rows per year
df.group_by('session_year').agg(pl.len().alias('nb_rows_per_year')).sort('session_year')

In [ ]:
# Number of rows per debate topic
df.group_by('debate_topic').agg(pl.len().alias('nb_rows_per_topic')).sort(
    'debate_topic'
)

## Merge data

In [ ]:
df = (
    df.join(
        pl.read_parquet('../data/Master_People.parquet'),
        on=['speaker_id', 'country'],
        how='left',
    )
    .join(
        pl.read_parquet('../data/Master_Affiliations.parquet').with_columns(
            pl.when(pl.col('start_date').is_not_null())
            .then(pl.col('start_date').str.to_date('%Y-%m-%d', strict=False))
            .otherwise(None)
            .alias('start_date'),
            pl.when(pl.col('end_date').is_not_null())
            .then(pl.col('end_date').str.to_date('%Y-%m-%d', strict=False))
            .otherwise(None)
            .alias('end_date'),
        ),
        on=['speaker_id', 'country'],
        how='left',
    )
    .filter(
        (pl.col('session_date') >= pl.col('start_date'))
        & (
            pl.col('end_date').is_null()
            | (pl.col('session_date') <= pl.col('end_date'))
        )
    )
    .unique()
    .join(
        pl.read_parquet('../data/Master_Organizations.parquet'),
        on=['org_id', 'country'],
        how='left',
    ).filter(pl.col('party_left_right_orientation').is_not_null())
)


df

## Filter data

In [ ]:
def get_results(filtered_df):

    return (
        filtered_df.with_columns(
            pl.struct(
                [
                    pl.col('session_date'),
                    pl.col('debate_topic'),
                    pl.col('speaker_id'),
                    pl.col('party_left_right_orientation'),
                    # pl.col('sentence_sentiment_value'),
                    # pl.col('sentence_sentiment_ana'),
                    (
                        pl.col('sentence_content_previous').fill_null('')
                        + ' '
                        + pl.col('sentence_content_current')
                        + ' '
                        + pl.col('sentence_content_next').fill_null('')
                    ).str.strip_chars(),
                ]
            ).alias('sentence_with_context')
        )
        .unique()
        .sort('session_date')
        .group_by([pl.col('entity_category'), pl.col('entity_content')])
        .agg(pl.col('sentence_with_context').alias('sentences_with_context'))
        .with_columns(pl.col('sentences_with_context').list.len().alias('nb_sentences'))
        .sort('nb_sentences', descending=True)
    )


In [ ]:
# Filter dataset to remove some historical conflicts
HISTORICAL_CONFLICTS = ['world war', 'cold war', 'winter war', 'homeland war']
df = df.filter(
    ~pl.col('entity_content').str.contains_any(
        HISTORICAL_CONFLICTS, ascii_case_insensitive=True
    ),
)

### Keywords

In [ ]:
# Look for a particular keyword in "sentence_content_current"
def search_keyword(keyword):
    pattern = rf'(?i){keyword}'
    return get_results(
        df.filter(
            pl.col('sentence_content_current').str.contains(pattern),
        )
    )

In [ ]:
# Mentions of "war"
search_keyword(r'\bwar\b')

In [ ]:
# Mentions of "conflict"
search_keyword('conflict')

### Entities

In [ ]:
# Look for a particular entity in "entity_content"
def search_entity(entity):
    pattern = rf'(?i){entity}'
    return get_results(df.filter(pl.col('entity_content').str.contains(pattern)))

In [ ]:
# Mentions of "ukrain"
search_entity('ukrain')

In [ ]:
# Mentions of "afghan"
search_entity('afghan')

### Keywords and entities

In [ ]:
# Look for a particular keyword associated to a particular entity
def search_keyword_for_entity(keyword, entity):
    pattern_entity = rf'(?i){entity}'
    pattern_keyword = rf'(?i){keyword}'

    return get_results(
        df.filter(
            pl.col('entity_content').str.contains(pattern_entity),
            pl.col('sentence_content_current').str.contains(pattern_keyword),
        )
    )


In [ ]:
# Look for mentions of "war" keyword and "afghan" entity
search_keyword_for_entity(r'\bwar\b', 'afghan')

In [ ]:
# Look for mentions of "war" keyword and "ukrain" entity
search_keyword_for_entity(r'\bwar\b', 'ukrain')